# Integrating AI Into Your Stack

In this session you will add AI to a real (small) system, step by step:

| Part | You learn | Why it matters |
| --- | --- | --- |
| 1. Text in, text out | Send a prompt, get an answer | The basic building block |
| 2. Structured responses | Get JSON your code can use | Free text can't go into a database |
| 3. Tools | Let the AI call your Python functions | The AI can use *your* data |
| 4. Full use case | Combine everything into a WhatsApp shop assistant | How it fits into a real workflow |

Run each cell with **Shift + Enter**, top to bottom. Change the text in quotes and run it again to try your own ideas.

**New to Python?** Each section links to a short example file in [`src/integrating_ai_into_your_stack/`](https://github.com/AI-Eswatini/integrating-ai-into-your-stack/tree/main/src/integrating_ai_into_your_stack) covering the Python it uses. In Colab, use the **GitHub** link; running locally, use the **Open locally** link.

**Before you start, add your API key**

1. Get a free key at [aistudio.google.com/api-keys](https://aistudio.google.com/api-keys) → **Create API key**.
2. In Colab, click the **key icon** (Secrets) in the left sidebar.
3. Add a secret named `GEMINI_API_KEY`, paste your key as the value, and turn on **Notebook access**.

Running locally instead? Set `GEMINI_API_KEY` as an environment variable, or paste it when the setup cell asks.

## Step 0: Setup

Two cells get you connected:

1. **Install** the latest `google-genai` library. In Colab, just run it. Running locally with uv? Skip it and run `uv sync` instead.
2. **Client**: reads your API key, connects to Gemini, and defines `ask()`, the one helper you'll use in Part 1.
   - **In Colab**, the key comes from **Secrets** (key icon in the left sidebar). Make sure **Notebook access** is on.
   - **Locally**, it uses the `GEMINI_API_KEY` environment variable, and only asks you to paste the key if that isn't set.

`ask(prompt)` sends your text to the model and returns its reply as a plain string. Everything in this notebook builds on that one call.

> **New to this Python?** Read and run these short examples:
> - `functions.py`: how `def`, arguments and `return` work in `ask()`. [Open on GitHub (Colab)](https://github.com/AI-Eswatini/integrating-ai-into-your-stack/blob/main/src/integrating_ai_into_your_stack/functions.py) · [Open locally](src/integrating_ai_into_your_stack/functions.py)

In [ ]:
# Colab: run this. Local uv users: skip it and run `uv sync` in a terminal instead.
%pip install -q -U google-genai

In [ ]:
import json
import os
from getpass import getpass

from google import genai

try:
    from google.colab import userdata  # Colab: read the key from Secrets
    api_key = userdata.get("GEMINI_API_KEY")
except ImportError:  # local: use the environment variable, or ask once
    api_key = os.environ.get("GEMINI_API_KEY") or getpass("Gemini API key: ")

client = genai.Client(api_key=api_key)
MODEL = "gemini-3.1-flash-lite"


def ask(prompt):
    """Send a prompt to the model and return its reply as text."""
    response = client.interactions.create(model=MODEL, input=prompt)
    return response.output_text

## Part 1: Text in, text out

### 1.1 Your first AI call

**What it does:** sends one sentence to the model and prints the answer.

**How it works:** `ask()` wraps `client.interactions.create(...)`. The model reads your `input` and returns text in `response.output_text`.

**Try it:** change the audience ("to a 10-year-old", "to a software engineer") and notice how the answer changes. The prompt *is* the program.

In [13]:
print(ask("Explain what AI is in two sentences, to someone who has never used a computer."))

AI is like a machine that has been taught to solve problems and make decisions by looking at patterns, much like a person learns from experience. It allows a device to act as if it has a mind of its own, helping humans by performing complex tasks or answering questions automatically.


### 1.2 Summarise

**Use case:** a nurse hands over a shift with a long, rambling paragraph of notes. The next nurse needs the urgent things first, not the whole story.

**How it works:** we put the data (the notes) and the instruction ("3 bullet points, most urgent first") into one prompt. Being specific about format and order gives far more useful answers than "summarise this".

**Try it:** ask for one line per bed, or for a list of tasks with times.

> **New to this Python?** Read and run these short examples:
> - `strings.py`: multi-line text with `"""` and joining text with `+`. [Open on GitHub (Colab)](https://github.com/AI-Eswatini/integrating-ai-into-your-stack/blob/main/src/integrating_ai_into_your_stack/strings.py) · [Open locally](src/integrating_ai_into_your_stack/strings.py)

In [14]:
notes = """
Busy afternoon on the ward. The man in bed 3 came in this morning complaining of chest pain that
started while he was walking to the taxi rank. His ECG was normal and he says the pain has eased,
but we are still waiting on his blood results, so the doctor wants him kept in and watched overnight.
His wife phoned twice asking when he can go home; I told her we will know more tomorrow. Bed 6 was
discharged at about 3pm after her family came to fetch her, and the bed still needs to be cleaned
before we can admit anyone else. The baby in bed 5 had a fever of 39.1 at 2pm, so we gave paracetamol
and the mother is staying with her. She needs her temperature rechecked at 8pm and the doctor must be
called if it is still above 38.5. Also, the ward is running low on gloves and I have put in an order,
but please follow up with stores tomorrow morning if it has not arrived.
"""

print(ask("Summarise these handover notes in 3 bullet points, most urgent first:\n" + notes))

Here is the summary of your handover notes, ranked by urgency:

*   **Bed 5:** The baby requires a temperature check at 8:00 PM; if the reading is above 38.5°C, the doctor must be notified immediately.
*   **Bed 3:** Patient is awaiting blood results following chest pain; he must remain in for overnight observation, and his wife is currently awaiting updates for tomorrow.
*   **Logistics:** Bed 6 needs to be cleaned for a new admission, and the glove supply order requires a follow-up with stores tomorrow morning if it hasn't arrived.


### 1.3 Sort messages (classification)

**Use case:** a shop gets many WhatsApp messages a day. Sorting them into orders, complaints and questions lets the right person handle each one.

**How it works:** we number all the messages and send them in **one** request, asking for one label per line. Limiting the answer to a fixed set of labels turns the AI into a *classifier*.

**Why batch?** One call for many messages instead of one call each: faster, cheaper, and less likely to hit rate limits. The model also sees all the messages together.

**The catch:** nothing *forces* the model to follow the format. It might say `Order.`, add a heading, or skip a line, and then the labels no longer line up with the messages. Part 2 fixes that.

> **New to this Python?** Read and run these short examples:
> - `list.py`: making and reading lists. [Open on GitHub (Colab)](https://github.com/AI-Eswatini/integrating-ai-into-your-stack/blob/main/src/integrating_ai_into_your_stack/list.py) · [Open locally](src/integrating_ai_into_your_stack/list.py)
> - `loops.py`: `for`, `enumerate` and `zip`, used to number and match messages. [Open on GitHub (Colab)](https://github.com/AI-Eswatini/integrating-ai-into-your-stack/blob/main/src/integrating_ai_into_your_stack/loops.py) · [Open locally](src/integrating_ai_into_your_stack/loops.py)

In [15]:
messages = [
    "Do you open on Sunday?",
    "pls send 2 breads and milk to number 14",
    "The eggs I bought yesterday were broken!",
]

numbered = "\n".join(f"{i}. {m}" for i, m in enumerate(messages, 1))

labels = ask(
    "Label each message ORDER, COMPLAINT or QUESTION. "
    "Reply with one label per line, in the same order, and nothing else:\n" + numbered
)

for label, message in zip(labels.strip().splitlines(), messages):
    print(label.strip(), "-", message)

QUESTION - Do you open on Sunday?
ORDER - pls send 2 breads and milk to number 14
COMPLAINT - The eggs I bought yesterday were broken!


## Part 2: Structured responses

Text is great for people, but **code needs data**. If you want to save an order to a database, add up a total, or route a message, you need fields you can rely on: `item`, `quantity`, `category`.

**How it works:** we pass a **JSON schema** with the request. The schema describes the exact shape of the answer: which fields, their types, and (with `enum`) which values are allowed. The model is constrained to return JSON that matches it, so `json.loads()` gives you a normal Python dict.

`ask_json(prompt, schema)` below works like `ask()`, but returns Python data instead of text.

> **New to this Python?** Read and run these short examples:
> - `json_data.py`: why the reply is text and how `json.loads()` turns it into data. [Open on GitHub (Colab)](https://github.com/AI-Eswatini/integrating-ai-into-your-stack/blob/main/src/integrating_ai_into_your_stack/json_data.py) · [Open locally](src/integrating_ai_into_your_stack/json_data.py)
> - `dict.py`: reading values by key, e.g. `result["category"]`. [Open on GitHub (Colab)](https://github.com/AI-Eswatini/integrating-ai-into-your-stack/blob/main/src/integrating_ai_into_your_stack/dict.py) · [Open locally](src/integrating_ai_into_your_stack/dict.py)

In [16]:
def ask_json(prompt, schema):
    """Send a prompt and get back Python data shaped like `schema`."""
    response = client.interactions.create(
        model=MODEL,
        input=prompt,
        response_format={"type": "text", "mime_type": "application/json", "schema": schema},
    )
    return json.loads(response.output_text)

### 2.1 Classify messages, reliably

**What changed from 1.3:** still **one batched call**, but now the schema asks for a list of results. The `enum` means `category` can *only* be `ORDER`, `COMPLAINT` or `QUESTION`, and each result carries the message `number` so we can match it back without trusting line order. We also get an `urgent` flag and a short `summary` in the same call.

`message_schema` describes **one** classified message; `batch_schema` wraps it in a list. Part 4 reuses `message_schema` on single messages.

**Why it matters:** the result is a list of dicts, so ordinary Python (`if`, `sorted`, filters) can act on it. No string cleaning needed.

> **New to this Python?** Read and run these short examples:
> - `loops.py`: the list comprehension that filters out complaints. [Open on GitHub (Colab)](https://github.com/AI-Eswatini/integrating-ai-into-your-stack/blob/main/src/integrating_ai_into_your_stack/loops.py) · [Open locally](src/integrating_ai_into_your_stack/loops.py)

In [17]:
messages = [
    "Do you open on Sunday?",
    "pls send 2 breads and milk to number 14",
    "The eggs I bought yesterday were broken!",
]

message_schema = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": ["ORDER", "COMPLAINT", "QUESTION"]},
        "urgent": {"type": "boolean"},
        "summary": {"type": "string", "description": "Under 10 words"},
    },
    "required": ["category", "urgent", "summary"],
}

batch_schema = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "number": {"type": "integer", "description": "The message number"},
                    **message_schema["properties"],
                },
                "required": ["number", *message_schema["required"]],
            },
        },
    },
    "required": ["results"],
}

numbered = "\n".join(f"{i}. {m}" for i, m in enumerate(messages, 1))
results = ask_json("Classify each of these customer messages:\n" + numbered, batch_schema)["results"]

for result in results:
    print(result["number"], result["category"], result["urgent"], "-", result["summary"])

# Now it's just data: plain Python can filter it
complaints = [r for r in results if r["category"] == "COMPLAINT"]
print("\nComplaints to handle today:", len(complaints))

1 QUESTION False - Inquiry about Sunday opening hours.
2 ORDER True - Request for bread and milk delivery.
3 COMPLAINT True - Report of damaged eggs received yesterday.

Complaints to handle today: 1


### 2.2 Extract an order

**Use case:** customers order in messy WhatsApp messages. We want a clean order the shop's system can store.

**How it works:** the schema describes an order: a list of `items` (each with a name and a whole-number quantity) plus a `delivery_day`. The `description` fields guide the model, e.g. "a dozen" becomes `12`.

**Notice:** the output is a dict, not a table of text. You can loop over it, save it, or send it to another system.

> **New to this Python?** Read and run these short examples:
> - `list.py`: a list of dicts, like the order's `items`. [Open on GitHub (Colab)](https://github.com/AI-Eswatini/integrating-ai-into-your-stack/blob/main/src/integrating_ai_into_your_stack/list.py) · [Open locally](src/integrating_ai_into_your_stack/list.py)

In [18]:
order_schema = {
    "type": "object",
    "properties": {
        "items": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "item": {"type": "string", "description": "Simple lowercase name, e.g. 'eggs', 'cooking oil'"},
                    "quantity": {"type": "integer", "description": "Number of units, e.g. 'a dozen eggs' = 12"},
                },
                "required": ["item", "quantity"],
            },
        },
        "delivery_day": {"type": "string", "description": "Day of the week, or 'unknown'"},
    },
    "required": ["items", "delivery_day"],
}

whatsapp = "Sawubona! pls send 2 bags of mealie meal, 1 oil and a dozen eggs, Friday is fine"

order = ask_json("Extract the order from this WhatsApp message:\n" + whatsapp, order_schema)

print(json.dumps(order, indent=2))
for line in order["items"]:
    print(f"{line['quantity']} x {line['item']}")

{
  "items": [
    {
      "item": "mealie meal",
      "quantity": 2
    },
    {
      "item": "cooking oil",
      "quantity": 1
    },
    {
      "item": "eggs",
      "quantity": 12
    }
  ],
  "delivery_day": "Friday"
}
2 x mealie meal
1 x cooking oil
12 x eggs


## Part 3: Tools (function calling)

The model knows a lot, but it does **not** know *your* data: today's stock, your prices, your customers. Tools fix that.

**How it works:**

1. You describe a Python function to the model (name, what it does, what inputs it takes).
2. When the model needs that information, it doesn't guess. It replies with a **function call**: "please run `check_stock` with `item='eggs'`".
3. **Your code** runs the function and sends the result back.
4. The model uses the real result to write its final answer.

The AI never touches your system directly. It only asks, and your code decides what actually runs. That's what makes this safe to plug into an existing stack.

> **New to this Python?** Read and run these short examples:
> - `functions.py`: `**arguments` and looking up a function by name in a dict. [Open on GitHub (Colab)](https://github.com/AI-Eswatini/integrating-ai-into-your-stack/blob/main/src/integrating_ai_into_your_stack/functions.py) · [Open locally](src/integrating_ai_into_your_stack/functions.py)
> - `json_data.py`: `json.dumps()`, used to send results back to the model. [Open on GitHub (Colab)](https://github.com/AI-Eswatini/integrating-ai-into-your-stack/blob/main/src/integrating_ai_into_your_stack/json_data.py) · [Open locally](src/integrating_ai_into_your_stack/json_data.py)

### 3.1 The shop's existing system

This is the "system" we're adding AI to: a stock list and a lookup function. In real life this would be a database query or an API call. The AI part doesn't change.

In [19]:
stock = {
    "mealie meal": {"price": 85.00, "in_stock": 10},
    "cooking oil": {"price": 45.00, "in_stock": 0},
    "eggs": {"price": 4.50, "in_stock": 30},
    "bread": {"price": 18.00, "in_stock": 20},
    "milk": {"price": 25.00, "in_stock": 8},
}


def check_stock(item):
    """Look up an item's price (in Rand, per unit) and how many are in stock."""
    return stock.get(item.lower(), {"error": f"We don't sell '{item}'"})


print(check_stock("eggs"))
print(check_stock("sugar"))

{'price': 4.5, 'in_stock': 30}
{'error': "We don't sell 'sugar'"}


### 3.2 Describe the tool and run the loop

**The tool description** is a small JSON schema, like in Part 2, but for the function's *inputs*. Write the `description` for the model: it uses it to decide *when* to call the tool.

**The loop** in `ask_with_tools()`:

- Ask the model, offering it the tools.
- If its reply contains function calls, run each one with our Python function and send the results back (linked with `previous_interaction_id` so the model remembers the conversation).
- Repeat until the model answers in plain text. We cap it at 5 rounds so a confused model can't loop forever.

The `print` inside the loop shows you every call the AI makes, so you can see what's happening.

In [20]:
check_stock_tool = {
    "type": "function",
    "name": "check_stock",
    "description": "Look up the price (Rand per unit) and number in stock for one shop item.",
    "parameters": {
        "type": "object",
        "properties": {
            "item": {"type": "string", "description": "Simple lowercase item name, e.g. 'eggs', 'cooking oil'"},
        },
        "required": ["item"],
    },
}

tools = [check_stock_tool]

# Maps the name the model uses to the real Python function
tool_functions = {"check_stock": check_stock}


def ask_with_tools(prompt):
    """Ask the model, running any tools it calls, until it gives a text answer."""
    interaction = client.interactions.create(model=MODEL, input=prompt, tools=tools)

    for _ in range(5):
        calls = [step for step in interaction.steps if step.type == "function_call"]
        if not calls:
            return interaction.output_text

        results = []
        for call in calls:
            print(f"  AI called {call.name}({call.arguments})")
            result = tool_functions[call.name](**call.arguments)
            results.append({
                "type": "function_result",
                "call_id": call.id,
                "name": call.name,
                "result": json.dumps(result),
            })

        interaction = client.interactions.create(
            model=MODEL,
            previous_interaction_id=interaction.id,
            input=results,
            tools=tools,
        )

    return "Stopped: too many tool calls."

### 3.3 Try it

Ask questions that need the shop's data. Watch the `AI called ...` lines: the model decides which items to look up, and the answer uses *our* prices, not made-up ones.

**Try it:** ask about something the shop doesn't sell, or a question that needs no lookup at all ("What are your opening hours?"). Does the model still call the tool?

In [21]:
print(ask_with_tools("Do you have cooking oil and eggs? How much are they?"))

  AI called check_stock({'item': 'cooking oil'})
  AI called check_stock({'item': 'eggs'})
We currently have eggs in stock, which are R4.50 each (we have 30 available). 

Unfortunately, we are currently out of cooking oil.


## Part 4: Full use case: a WhatsApp shop assistant

Now we connect everything into one workflow, the way you'd add AI to a real system:

1. **Classify and extract in one call** (structured output, Part 2). We merge the two schemas from Part 2 so a single request returns the category *and*, for orders, the items. Fewer calls means faster and cheaper.
2. **Check stock and price with our own code.** No AI needed here: once the data is structured, plain Python is faster, cheaper and always correct at maths.
3. **Draft a friendly reply** with `ask()` (Part 1).
4. For questions, **answer with tools** so the reply uses real data (Part 3).
5. **A person approves** before anything is sent or saved. AI drafts; humans decide.

Each step uses the simplest tool that works. That's the main lesson: AI handles the messy language, your code handles the rules.

> **New to this Python?** Read and run these short examples:
> - `loops.py`: `if`/`else` and loops, used to check stock and add up the total. [Open on GitHub (Colab)](https://github.com/AI-Eswatini/integrating-ai-into-your-stack/blob/main/src/integrating_ai_into_your_stack/loops.py) · [Open locally](src/integrating_ai_into_your_stack/loops.py)

In [ ]:
# One schema for both jobs: the classification fields plus the order fields
reply_schema = {
    "type": "object",
    "properties": {**message_schema["properties"], **order_schema["properties"]},
    "required": message_schema["required"],
}


def handle_message(message):
    """Turn one customer message into a draft reply for the shop owner to check."""
    info = ask_json(
        "Classify this customer message. If it is an order, also fill in the items and delivery day:\n" + message,
        reply_schema,
    )
    print(f"[{info['category']}] urgent={info['urgent']} - {info['summary']}")

    if info["category"] == "ORDER":
        # Our own code checks stock and adds up the total
        lines, total = [], 0
        for line in info.get("items", []):
            product = check_stock(line["item"])
            if "error" in product or product["in_stock"] < line["quantity"]:
                lines.append(f"{line['item']}: NOT AVAILABLE")
            else:
                cost = product["price"] * line["quantity"]
                total += cost
                lines.append(f"{line['quantity']} x {line['item']}: R{cost:.2f}")
        summary = "\n".join(lines) + f"\nTotal: R{total:.2f}, delivery: {info.get('delivery_day', 'unknown')}"
        print(summary)

        return ask("Write a short, friendly WhatsApp reply confirming this order. "
                   "Mention anything not available. Order details:\n" + summary)

    if info["category"] == "COMPLAINT":
        return ask("Write a short, kind WhatsApp reply apologising and saying the owner will call back:\n" + message)

    return ask_with_tools("Answer this customer's WhatsApp question briefly. "
                          "If you don't know (e.g. opening hours), say the owner will reply soon:\n" + message)

### 4.1 Handle a message

**What it does:** runs one customer message through `handle_message()` and asks you to approve the draft, just like the shop owner would. An order takes just **2 AI calls**: one to classify and extract, one to write the reply.

Type `y` to approve. Approved replies go into `outbox`, which is where your real system would take over (send via WhatsApp, save to a database, etc.).

**Try it:** swap in one of the other messages in the comment to see the complaint and question paths.

In [ ]:
message = "Sawubona! pls send 2 bags of mealie meal, 1 oil and a dozen eggs, Friday is fine"
# Or try: "The bread I bought this morning was already mouldy!!"
# Or try: "How much is milk?"

outbox = []
draft = handle_message(message)
print("\nDraft reply:\n" + draft)

if input("Send this reply? (y/n) ").strip().lower() == "y":
    outbox.append({"to_message": message, "reply": draft})

print(f"\n{len(outbox)} reply approved and ready to send.")

## Recap: which technique when?

| Technique | Use it when | In this notebook |
| --- | --- | --- |
| `ask()` (text) | A person will read the result | Summaries, reply drafts |
| `ask_json()` (structured) | Code will use the result | Classifying, extracting orders |
| `ask_with_tools()` (tools) | The AI needs your live data to answer | Stock and price questions |
| Plain Python | The rule is clear (maths, lookups, checks) | Totals, stock checks |
| A human check | The action matters (money, customers, health) | Approving replies |

## Your turn

Pick a task from your own work and build a small version of it. Some ideas:

- **Text:** draft a weekly report from your meeting notes.
- **Structured:** turn job applications or invoices into rows with fixed fields.
- **Tools:** write a function that reads your own data (a CSV, a list) and let the AI answer questions about it.

Start with `ask()`, then move to `ask_json()` once you know which fields you need.

In [24]:
print(ask("Write your own request here"))

Since I am an AI, I don't have personal needs, but I do have a "functional wish list" that helps me provide better service to you. 

If you'd like to help me be more effective, here is my request:

**"Please give me a specific persona, a clear goal, and a preferred format for your next request."**

***

**Why this helps:**
*   **The Persona:** If you tell me to "act as a senior software engineer" or "act as a creative writing tutor," I can adjust my tone, vocabulary, and expertise level to match exactly what you need.
*   **The Goal:** If I know *why* you are asking (e.g., "I need this to convince my boss" vs. "I need this to study for an exam"), I can prioritize the information that is most relevant to your success.
*   **The Format:** If you need a bulleted list, a professional email, a poem, or a table, telling me upfront saves you time on formatting and editing.

**Would you like to try it now? Give me a topic, and tell me who I should be while explaining it to you.**
